# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cassandragoto10-maker/Flyrank-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
import sys
import subprocess

# Install the packages needed for the warehouse notebook
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"],
    check=True
)

from google.colab import userdata

HF_TOKEN = userdata.get("HF_token")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found. Check Colab Secrets.")

print("HF_TOKEN found.")

HF_TOKEN found.


In [3]:
from huggingface_hub import login

login(token=HF_TOKEN, add_to_git_credential=False)

print("Hugging Face login successful.")

Hugging Face login successful.


In [4]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of Analysis + Time Window

**Unit of analysis:** One row represents the daily performance observation for one content page for one client on one reporting date.

**Time window:** I use March 2026 as the analysis month. This is a mid-panel month rather than the June 2026 `_sample`, which FlyRank identifies as the final month and a sealed test month.

In [7]:
con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute("CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN ?)", [HF_TOKEN])

print("DuckDB Hugging Face authentication configured.")

DuckDB Hugging Face authentication configured.


In [9]:
path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

df_march = con.execute(f"""
    SELECT *
    FROM read_parquet('{path}')
""").df()

print("Rows:", len(df_march))
print("Columns:", len(df_march.columns))

df_march.head()

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 401)

In [10]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Downloaded file:", march_file)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Downloaded file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [11]:
df_march = con.execute(f"""
    SELECT *
    FROM read_parquet('{march_file}')
""").df()

print("Rows:", len(df_march))
print("Columns:", len(df_march.columns))

df_march.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378
Columns: 31


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [12]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Downloaded file:", march_file)

Downloaded file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: Feature / Label / Context / Excluded

**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, and `scroll_events`. These are performance signals that could be known at the point when a content refresh decision is made.

**Label:** A future content-performance decline indicator. The March warehouse slice does not contain this future outcome directly, so it would need to be constructed using a later outcome window.

**Context:** `client_hash_id`, `content_hash_id`, `report_date`, `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`, and `month`. These fields describe the observation and its data coverage rather than being direct performance features.

**Excluded:** `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, and `ai_other` are excluded from the first five-feature frame. They may provide additional context about AI traffic, but they are not needed for this initial contract and would expand the feature set beyond the five-feature limit.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Verify the grain

The expected grain is one client-content-date combination per row. I will check whether the combination of `client_hash_id`, `content_hash_id`, and `report_date` is unique.


In [13]:
grain_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS unique_client_content_dates,
        COUNT(*) - COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS duplicate_rows
    FROM read_parquet('{march_file}')
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_client_content_dates,duplicate_rows
0,9841378,9841378,0


### Query 2 — Row count and date span

The March 2026 partition contains 9,841,378 rows. I will verify the number of rows and the earliest and latest reporting dates in the partition.

In [14]:
date_check = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{march_file}')
""").df()

date_check

,row_count,earliest_date,latest_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — GSC data availability

I will check how many March 2026 rows have Google Search Console data available. The availability condition is written explicitly as `IS TRUE` so that only rows where availability is confirmed are counted.

In [15]:
availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_available_rows
    FROM read_parquet('{march_file}')
""").df()

availability_check

,total_rows,gsc_available_rows,gsc_not_available_rows
0,9841378,3611061,6230317


In [16]:
print(df_march.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


### Five-feature frame

For the content refresh lane, I will use five performance features from the March 2026 data: `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, and `scroll_events`.

These features describe search visibility, search traffic, website traffic and user engagement. They are used as inputs for a future refresh decision rather than as the outcome being predicted.

In [17]:
features = [
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

feature_frame = df_march[features].copy()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head()

Feature frame shape: (9841378, 6)


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>


### When are these features available?

- **gsc_impressions** — knowable at the decision moment because the value is recorded in the warehouse as Google Search Console performance data for the reporting period.
- **gsc_clicks** — knowable at the decision moment because the value is recorded as Google Search Console clicks for the reporting period.
- **gsc_avg_position** — knowable at the decision moment because the average search position is recorded in the available GSC performance data.
- **ga4_sessions** — knowable at the decision moment because the number of GA4 sessions for the page is recorded in the warehouse.
- **scroll_events** — knowable at the decision moment because recorded scroll events are available in the warehouse for the reporting period.

These features describe information available from the selected reporting period. They should only be used to predict or rank a future outcome if the modelling setup keeps the feature window before that outcome window.

In [18]:
feature_frame.describe().T

,count,mean,std,min,25%,50%,75%,max
gsc_impressions,9841378.0,28.518119,155.926569,0.0,0.0,0.0,6.0,40084.0
gsc_clicks,9841378.0,0.083508,0.781434,0.0,0.0,0.0,0.0,274.0
gsc_avg_position,3611061.0,15.826651,19.856034,0.0,3.74212,7.5,20.2,498.0
ga4_sessions,6822637.0,0.190514,1.96875,0.0,0.0,0.0,0.0,792.0
scroll_events,6822637.0,0.032261,0.413166,0.0,0.0,0.0,0.0,254.0


In [19]:
feature_frame.isna().sum()

,0
content_hash_id,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,6230317
ga4_sessions,3018741
scroll_events,3018741


In [20]:
print("Available March columns:")
for i, col in enumerate(df_march.columns, 1):
    print(i, col)

Available March columns:
1 report_date
2 client_hash_id
3 content_hash_id
4 client_has_gsc
5 client_has_ga4
6 gsc_data_available
7 ga4_data_available
8 gsc_impressions
9 gsc_clicks
10 gsc_sum_position
11 gsc_avg_position
12 ga4_pageviews
13 ga4_sessions
14 ga4_users
15 ga4_engaged_sessions
16 ga4_total_engagement_sec
17 sessions_organic
18 sessions_direct
19 sessions_referral
20 sessions_social
21 sessions_paid
22 sessions_ai
23 ai_chatgpt
24 ai_perplexity
25 ai_gemini
26 ai_copilot
27 ai_claude
28 ai_meta
29 ai_other
30 scroll_events
31 month


### Five-feature frame

For the content refresh lane, I will use five performance features from the March 2026 data: `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, and `scroll_events`.

- `gsc_impressions` — available at the reporting period because it records observed search impressions.
- `gsc_clicks` — available at the reporting period because it records observed search clicks.
- `gsc_avg_position` — available at the reporting period when GSC data is available.
- `ga4_sessions` — available at the reporting period when GA4 data is available.
- `scroll_events` — available at the reporting period when GA4 data is available.

These features describe observed search visibility, search traffic, website traffic and engagement. They would be used as inputs for a future refresh decision rather than as the future outcome being predicted.

In [21]:
feature_frame = df_march[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "scroll_events"
    ]
].copy()

feature_frame.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>


### Leakage trap

The prepared Content Refresh dataset defines decline using `trend_direction`, but this field is not present in the March 2026 warehouse partition used for this contract. I therefore do not construct a leakage score from it here. A proper leakage test requires the future outcome label to be available alongside the features, and using a future-derived field as an input would make the evaluation invalid.

The feature frame contains the five selected performance features from the March 2026 slice. Missing values are visible because GSC and GA4 data are not available for every observation. The features are kept in their observed form here rather than filling missing values, since the purpose of this section is to document what is available at the decision point.

## 4. Data limits

This March 2026 slice contains observations with different levels of data availability. GSC data is available for 3,611,061 of the 9,841,378 rows, while 6,230,317 rows do not have GSC data available. GA4 data is also not available for every row. This limits which search and analytics signals can be used for individual observations.

The March partition represents one historical month. It can describe observed performance during that period, but it cannot by itself establish whether a page will decline in a future period. A future outcome window is needed when constructing a predictive target.

The data is anonymized, so the analysis cannot identify individual clients or use client-specific information outside the available hashed identifiers.

### Data limits

This March 2026 slice contains observations for different levels of data availability. GSC data is not available for every row, and GA4 data is also not available for every row. This limits which search and analytics signals can be used for individual observations.

The March partition also represents one historical month. It can describe observed performance during that period, but it cannot by itself establish whether a page will decline in a future period. A future outcome window is needed when constructing a predictive target.

The data is anonymized, so the analysis cannot identify individual clients or use client-specific information outside the available hashed identifiers.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.